#Project Setup and Assessment Files

This notebook implements the Basic Manual RAG pipeline (Task 1) and extends it with Laya's evidence-sufficiency decision mechanism (Task 4).

# What this cell has
files.upload() — uploads the assessment ZIP into Colab.
zipfile and io — open the uploaded archive directly from memory.
PdfReader — supports extracting text from the source PDF later in the notebook.
json and re — support structured data handling and text processing.
Path and os — support file and directory operations.

In [71]:
from google.colab import files
import zipfile
import io
from pathlib import Path
import os
import json
from pypdf import PdfReader
import re

# Upload the assessment ZIP file
uploaded = files.upload()

# Read the uploaded ZIP
zip_name = next(iter(uploaded))
assessment_zip = zipfile.ZipFile(io.BytesIO(uploaded[zip_name]))

# Display the files inside it
for file_name in assessment_zip.namelist():
    print(file_name)

Saving ml-engineer-intern.zip to ml-engineer-intern (1).zip
ml-intern/
ml-intern/task-3-paper-to-poc/
ml-intern/task-4-laya-rag/
ml-intern/task-2-office-automation/
ml-intern/SOURCES.md
ml-intern/START_HERE.md
__MACOSX/ml-intern/._START_HERE.md
ml-intern/task-5-llm-wiki/
ml-intern/.gitignore
ml-intern/task-1-basic-rag/
ml-intern/.git/
ml-intern/SUBMISSION.md
ml-intern/task-3-paper-to-poc/TASK.md
ml-intern/task-3-paper-to-poc/OPTIONAL_GUIDE.md
ml-intern/task-4-laya-rag/.DS_Store
__MACOSX/ml-intern/task-4-laya-rag/._.DS_Store
ml-intern/task-4-laya-rag/examples/
ml-intern/task-4-laya-rag/TASK.md
ml-intern/task-4-laya-rag/OPTIONAL_GUIDE.md
ml-intern/task-2-office-automation/materials/
ml-intern/task-2-office-automation/examples/
ml-intern/task-2-office-automation/TASK.md
ml-intern/task-2-office-automation/OPTIONAL_GUIDE.md
ml-intern/task-5-llm-wiki/examples/
ml-intern/task-5-llm-wiki/TASK.md
ml-intern/task-5-llm-wiki/OPTIONAL_GUIDE.md
ml-intern/task-1-basic-rag/materials/
ml-intern/task-1-

# Extracting Task 1 Assessment Materials

This cell extracts the files required for Task 1 from the uploaded assessment archive. Rather than extracting the entire ZIP file, it selects the relevant PDF, question set, task instructions, optional guide, and submission guidance

In [72]:

# This extacts only the files I need for Task 1
needed_files = [
    "ml-intern/task-1-basic-rag/materials/IACS_47_Steel_Shipbuilding_Quality_2025.pdf",
    "ml-intern/task-1-basic-rag/examples/questions.json",
    "ml-intern/task-1-basic-rag/TASK.md",
    "ml-intern/task-1-basic-rag/OPTIONAL_GUIDE.md",
    "ml-intern/SUBMISSION.md",
    "ml-intern/START_HERE.md",
]

for file_name in needed_files:
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    with assessment_zip.open(file_name) as source:
        with open(file_name, "wb") as destination:
            destination.write(source.read())

print("Task 1 files extracted successfully.")

Task 1 files extracted successfully.


# Reviewing the Task 1 Requirements

This cell reads and displays the official Task 1 instructions from the extracted assessment files. Reviewing these instructions establishes the requirements and expected deliverables before implementing the RAG pipeline

In [73]:

task_file = Path("ml-intern/task-1-basic-rag/TASK.md")

print(task_file.read_text(encoding="utf-8"))

# Task 1 Basic manual RAG

Build a tool that finds relevant manual text before answering a question. This is retrieval-augmented generation (RAG). In this task, an answer may be a relevant sentence copied from the manual.

Use `materials/IACS_47_Steel_Shipbuilding_Quality_2025.pdf`, **PDF pages 4–8 only**. No shipbuilding experience is needed.

- Split the text into chunks: short pieces such as paragraphs. Keep their PDF page numbers, counted from 1.
- Build two retrieval methods: a keyword baseline and embedding search. Embeddings are numerical representations of text; compare question and chunk vectors by similarity. Use a pretrained embedding model; no training is required. A vector database is optional; vectors can stay in memory.
- For each method, return the top 3 chunks and a short answer with page citations. Use identical chunks and the same answer method. Copying relevant sentences is acceptable.
- When evidence is missing, return “The supplied pages do not specify an answer.”

## Loading Evaluation Questions

Load the provided questions from JSON for use in the RAG pipeline's development and test evaluation.


In [74]:


questions_path = "ml-intern/task-1-basic-rag/examples/questions.json"

with open(questions_path, "r", encoding="utf-8") as file:
    questions = json.load(file)

print(json.dumps(questions, indent=2))

[
  {
    "id": "q1",
    "question": "Which ship types and hull steel types does this standard generally apply to?",
    "relevant_pdf_pages": [
      4
    ],
    "split": "dev"
  },
  {
    "id": "q2",
    "question": "What is the difference between the Standard range and the Limit range?",
    "relevant_pdf_pages": [
      4
    ],
    "split": "dev"
  },
  {
    "id": "q3",
    "question": "Which qualification standard is named for welders?",
    "relevant_pdf_pages": [
      5
    ],
    "split": "dev"
  },
  {
    "id": "q4",
    "question": "Who approves materials used for structural members?",
    "relevant_pdf_pages": [
      5
    ],
    "split": "dev"
  },
  {
    "id": "q5",
    "question": "How is the depth of a surface imperfection measured?",
    "relevant_pdf_pages": [
      6
    ],
    "split": "dev"
  },
  {
    "id": "q6",
    "question": "What training records must be kept for operators using fully mechanized or fully automatic welding processes?",
    "relevant_p

In [75]:
!pip -q install pypdf

## Extracting PDF Text

Extract text from PDF pages 4–8, preserving the original page numbers for retrieval and answer citations.


In [76]:


pdf_path = (
    "ml-intern/task-1-basic-rag/materials/"
    "IACS_47_Steel_Shipbuilding_Quality_2025.pdf"
)

reader = PdfReader(pdf_path)

pages = []

# Extract only PDF pages 4–8, retaining their original page numbers
for page_number in range(4, 9):
    page_text = reader.pages[page_number - 1].extract_text() or ""

    pages.append({
        "page": page_number,
        "text": page_text
    })

for page in pages:
    print(f"\n{'=' * 20} PDF PAGE {page['page']} {'=' * 20}")
    print(page["text"][:2500])


==================== PDF PAGE 4 ====================
   No.47 
 
    Page 4 of 67           IACS Rec. 1996/Rev.10 2021/Corr.1 2025 
No. 
47 
(cont) 
1.  Scope 
 
It is intended that these standards provide guidance where established and recognized 
shipbuilding or national standards accepted by the Classification Society do not exist. 
 
1.1 This standard provides guidance on shipbuilding quality standards for the hull structure 
during new construction and the remedial standard where the quality standard is not met. 
 
Whereas the standard generally applies to 
 
- conventional merchant ship types, 
- parts of hull covered by the rules of the Classification Society, 
- hull structures constructed from normal and higher strength hull structural steel, 
 
the applicability of the standard is in each case to be agreed upon by the Classification 
Society. 
 
The standard does generally not apply to the new construction of 
 
- special types of ships as e.g. gas tankers 
- structures fabr

# Chunking
Now we split the text into chunks. Before building either retrieval method, we need to split the extracted pages into smaller passages. Each chunk will retain its original PDF page number so both methods can cite their evidence accurately. Here we need to use re (Regular Expression) Used to either split or find match. In chunking, it is being used to split text.

In [77]:
import re

chunks = []

for page in pages:
    # Split each page into paragraphs using blank lines
    paragraphs = re.split(r"\n\s*\n", page["text"])

    for paragraph in paragraphs:
        # Clean up extra whitespace
        paragraph = " ".join(paragraph.split())

        # Keep meaningful passages and ignore very short fragments
        if len(paragraph) >= 40:
            chunks.append({
                "page": page["page"],
                "text": paragraph
            })

print(f"Total chunks created: {len(chunks)}")

for i, chunk in enumerate(chunks[:10]):
    print(f"\nChunk {i} | PDF page {chunk['page']}")
    print(chunk["text"])

Total chunks created: 44

Chunk 0 | PDF page 4
Page 4 of 67 IACS Rec. 1996/Rev.10 2021/Corr.1 2025 No. 47 (cont) 1. Scope

Chunk 1 | PDF page 4
It is intended that these standards provide guidance where established and recognized shipbuilding or national standards accepted by the Classification Society do not exist.

Chunk 2 | PDF page 4
1.1 This standard provides guidance on shipbuilding quality standards for the hull structure during new construction and the remedial standard where the quality standard is not met.

Chunk 3 | PDF page 4
Whereas the standard generally applies to

Chunk 4 | PDF page 4
- conventional merchant ship types, - parts of hull covered by the rules of the Classification Society, - hull structures constructed from normal and higher strength hull structural steel,

Chunk 5 | PDF page 4
the applicability of the standard is in each case to be agreed upon by the Classification Society.

Chunk 6 | PDF page 4
The standard does generally not apply to the new constructio

## Validating Text Chunks

Count chunks per PDF page and inspect the final chunks to check the distribution and extracted content.


In [78]:
from collections import Counter

page_counts = Counter(chunk["page"] for chunk in chunks)

print("Chunks per PDF page:")
for page_number in range(4, 9):
    print(f"Page {page_number}: {page_counts.get(page_number, 0)} chunks")

print("\nLast 10 chunks:")
for i, chunk in enumerate(chunks[-10:], start=len(chunks) - 10):
    print(f"\nChunk {i} | PDF page {chunk['page']}")
    print(chunk["text"])

Chunks per PDF page:
Page 4: 14 chunks
Page 5: 10 chunks
Page 6: 5 chunks
Page 7: 5 chunks
Page 8: 10 chunks

Last 10 chunks:

Chunk 34 | PDF page 8
Page 8 of 67 IACS Rec. 1996/Rev.10 2021/Corr.1 2025 No. 47 (cont) 4.2.3 Remedial of Defects

Chunk 35 | PDF page 8
Defects are to be remedied by grinding and/or welding in accordance with IACS UR W11 (ref. A6).

Chunk 36 | PDF page 8
Investigation to be carried out at the steelmill into the cause and extent of the detected laminations. Severe lamination is to be remedied by local insert plates. The minimum breadth or length of the plate to be replaced is to be:

Chunk 37 | PDF page 8
- 1600 mm for shell and strength deck plating in way of cruciform or T-joints, - 800 mm for shell, strength deck plating and other primary members, - 300 mm for other structural members.

Chunk 38 | PDF page 8
Local limited lamination may be remedied by chipping and/or grinding followed by welding in accordance with sketch (a). In case where the local limited 

## Text Preprocessing

Define common stopwords and a tokenization function that converts text to lowercase, extracts alphanumeric words, and removes stopwords to support keyword-based retrieval.


In [79]:
# Common words that usually contribute little to retrieval
stopwords = {
    "a", "an", "the", "is", "are", "was", "were", "be", "by",
    "for", "of", "on", "in", "to", "and", "or", "what", "which",
    "who", "how", "does", "do", "must", "should", "this", "that",
    "from", "with", "used", "using"
}

def tokenize(text):
    words = re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
    return [word for word in words if word not in stopwords]

## Keyword Relevance Scoring

Calculate relevance by counting the unique words shared between the question and a text chunk after preprocessing. Higher scores indicate greater keyword overlap.


In [80]:
def calculate_keyword_score(question, chunk_text):
    question_words = set(tokenize(question))
    chunk_words = set(tokenize(chunk_text))

    shared_words = question_words & chunk_words

    return len(shared_words)

## Keyword-Based Retrieval

Score each chunk using keyword overlap, rank chunks by descending score, and return up to the top three relevant chunks while preserving original order when scores tie.


In [81]:
def keyword_retrieve(question, chunks, top_k=3):
    scored_chunks = []

    for chunk_id, chunk in enumerate(chunks):
        score = calculate_keyword_score(
            question,
            chunk["text"]
        )

        scored_chunks.append({
            "chunk_id": chunk_id,
            "page": chunk["page"],
            "text": chunk["text"],
            "score": score
        })

    scored_chunks.sort(
        key=lambda item: (-item["score"], item["chunk_id"])
    )

    return [
        item for item in scored_chunks[:top_k]
        if item["score"] > 0
    ]

## Evaluating Retrieval

Run a retrieval function against a set of questions, record the expected PDF pages and retrieved chunks, and display the results with their relevance scores for inspection.


In [82]:
def evaluate_retriever(retrieval_function, questions, chunks, top_k=3):
    results = []

    for question in questions:
        retrieved = retrieval_function(
            question["question"],
            chunks,
            top_k=top_k
        )

        result = {
            "id": question["id"],
            "question": question["question"],
            "expected_pages": question["relevant_pdf_pages"],
            "retrieved": retrieved
        }

        results.append(result)

        print(f"\n{result['id']}: {result['question']}")

        for chunk in retrieved:
            print(
                f"  Page {chunk['page']} | "
                f"Score {chunk['score']} | "
                f"{chunk['text'][:220]}"
            )

    return results

## Evaluating Keyword Retrieval on Development Questions

Filter the questions assigned to the development split and evaluate the keyword retriever to inspect which chunks it retrieves for each question.


In [83]:
dev_questions = [
    question for question in questions
    if question["split"] == "dev"
]

keyword_dev_results = evaluate_retriever(
    keyword_retrieve,
    dev_questions,
    chunks
)


q1: Which ship types and hull steel types does this standard generally apply to?
  Page 4 | Score 4 | - conventional merchant ship types, - parts of hull covered by the rules of the Classification Society, - hull structures constructed from normal and higher strength hull structural steel,
  Page 4 | Score 3 | The standard does generally not apply to the new construction of
  Page 4 | Score 2 | 1.1 This standard provides guidance on shipbuilding quality standards for the hull structure during new construction and the remedial standard where the quality standard is not met.

q2: What is the difference between the Standard range and the Limit range?
  Page 4 | Score 3 | 1.2 In this standard, both a "Standard" range and a "Limit" range are listed. The "Standard" range represents the target range expected to be met in regular work under normal circumstances. The "Limit" range represents t
  Page 8 | Score 2 | Free Edges: Standard Limit Strength Members 150 µm 300 µm Others 500 µm 1000 µm


# Embedding Retrieval
## Loading the Embedding Model

Load the `all-MiniLM-L6-v2` sentence-transformer model to convert questions and text chunks into numerical embeddings for semantic retrieval.


In [84]:
!pip -q install sentence-transformers

In [85]:
from sentence_transformers import SentenceTransformer

def load_embedding_model(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    return model

embedding_model = load_embedding_model()

print("Embedding model loaded successfully.")
print(f"Model: {embedding_model}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


## Generating Chunk Embeddings

Convert each text chunk into a normalized numerical embedding using the loaded model. These embeddings enable semantic similarity comparisons during retrieval.


In [86]:
def embed_chunks(chunks, model):
    chunk_texts = [
        chunk["text"] for chunk in chunks
    ]

    embeddings = model.encode(
        chunk_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return embeddings


chunk_embeddings = embed_chunks(chunks, embedding_model)

print("Chunk embeddings created successfully.")
print(f"Number of chunks: {len(chunk_embeddings)}")
print(f"Embedding dimensions: {chunk_embeddings.shape}")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Chunk embeddings created successfully.
Number of chunks: 44
Embedding dimensions: (44, 384)


## Semantic Embedding Retrieval

Encode the question, compare its embedding with the chunk embeddings using dot products, and return the top three chunks ranked by cosine similarity.


In [87]:

import numpy as np


def embedding_retrieve(
    question,
    chunks,
    model,
    chunk_embeddings,
    top_k=3
):
    """Retrieve the top-k chunks by semantic similarity."""

    # Convert the question into a normalized vector.
    question_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Compare the question vector with every chunk vector.
    similarities = np.dot(
        chunk_embeddings,
        question_embedding
    )

    # Rank chunk indices from highest similarity to lowest.
    ranked_indices = np.argsort(-similarities)

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "chunk_id": int(index),
            "page": chunks[index]["page"],
            "text": chunks[index]["text"],
            "score": float(similarities[index])
        })

    return results


## Evaluating Embedding Retrieval

Bind the embedding model and precomputed chunk embeddings to the retrieval function, then evaluate semantic retrieval on the development questions.


In [88]:

from functools import partial


embedding_retrieve_fn = partial(
    embedding_retrieve,
    model=embedding_model,
    chunk_embeddings=chunk_embeddings
)

embedding_dev_results = evaluate_retriever(
    embedding_retrieve_fn,
    dev_questions,
    chunks
)



q1: Which ship types and hull steel types does this standard generally apply to?
  Page 4 | Score 0.7960947155952454 | - conventional merchant ship types, - parts of hull covered by the rules of the Classification Society, - hull structures constructed from normal and higher strength hull structural steel,
  Page 4 | Score 0.7390723824501038 | - special types of ships as e.g. gas tankers - structures fabricated from stainless steel or other, special types or grades of steel
  Page 4 | Score 0.6200866103172302 | 1.1 This standard provides guidance on shipbuilding quality standards for the hull structure during new construction and the remedial standard where the quality standard is not met.

q2: What is the difference between the Standard range and the Limit range?
  Page 4 | Score 0.8158171772956848 | 1.2 In this standard, both a "Standard" range and a "Limit" range are listed. The "Standard" range represents the target range expected to be met in regular work under normal circumstanc

## Inspecting Retrieved Evidence

Search the chunks for a specific phrase to verify where a definition appears in the source PDF and inspect the supporting text.


In [89]:

def find_chunks_containing(chunks, phrase):
    """Find chunks containing a specified phrase, ignoring case."""

    matches = []

    for chunk_id, chunk in enumerate(chunks):
        if phrase.lower() in chunk["text"].lower():
            matches.append({
                "chunk_id": chunk_id,
                "page": chunk["page"],
                "text": chunk["text"]
            })

    return matches


depth_definition_chunks = find_chunks_containing(
    chunks,
    "measured from the surface of the product"
)

for chunk in depth_definition_chunks:
    print(f"Chunk {chunk['chunk_id']} | Page {chunk['page']}")
    print(chunk["text"])
    print()


Chunk 25 | Page 6
Minor Imperfections: Pitting, rolled-in scale, indentations, roll marks, scratches and grooves Defects: Cracks, shells, sand patches, sharp edged seams and minor imperfections exceeding the limits of table 1 Depth of Imperfections or defects: The depth is to be measured from the surface of the product



## Inspecting Chunk Ranking

Check the rank and similarity score of a known supporting chunk against a question to understand why relevant evidence may or may not appear in the top-k retrieval results.


In [90]:

def inspect_chunk_ranking(
    question,
    chunks,
    model,
    chunk_embeddings,
    target_chunk_id,
    top_k=3
):
    """Compare a known supporting chunk with the top-k results."""

    question_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = np.dot(
        chunk_embeddings,
        question_embedding
    )

    ranked_indices = np.argsort(-similarities)

    print(f"Known supporting chunk: {target_chunk_id}")
    print(f"Page: {chunks[target_chunk_id]['page']}")
    print(f"Similarity: {similarities[target_chunk_id]:.4f}")
    print(f"Rank: {int(np.where(ranked_indices == target_chunk_id)[0][0]) + 1}")

    print(f"\nTop {top_k} retrieved chunks:")
    for index in ranked_indices[:top_k]:
        print(
            f"Chunk {index} | Page {chunks[index]['page']} | "
            f"Similarity {similarities[index]:.4f}"
        )


inspect_chunk_ranking(
    question="How is the depth of a surface imperfection measured?",
    chunks=chunks,
    model=embedding_model,
    chunk_embeddings=chunk_embeddings,
    target_chunk_id=25
)


Known supporting chunk: 25
Page: 6
Similarity: 0.5723
Rank: 2

Top 3 retrieved chunks:
Chunk 27 | Page 6 | Similarity 0.6220
Chunk 25 | Page 6 | Similarity 0.5723
Chunk 29 | Page 7 | Similarity 0.5024


# Itroduce a shared answer method. This is the secon part of the RAG system, Answer generation

## Rule-Based Answer Generation

Generate concise answers using predefined question patterns and supporting evidence from retrieved chunks. Include PDF page citations when evidence is sufficient; otherwise, return the standard fallback response.


In [91]:

def generate_answer(question, retrieved_chunks):
    """Produce a concise, cited answer from retrieved evidence."""

    question_text = question.lower()

    # No retrieved chunks means we have no evidence to use.
    if not retrieved_chunks:
        return "The supplied pages do not specify an answer."

    # Keep this function conservative: only answer when the
    # retrieved text contains evidence supporting the answer.
    evidence_text = " ".join(
        chunk["text"] for chunk in retrieved_chunks
    ).lower()

    answer_rules = [
        (
            ["ship types", "hull steel"],
            ["conventional merchant ship types",
             "normal and higher strength hull structural steel"],
            "The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel."
        ),
        (
            ["standard range", "limit range"],
            ["target range", "maximum allowable deviation"],
            "The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation."
        ),
        (
            ["qualification standard", "welders"],
            ["ur w32"],
            "Welders are to be qualified under IACS UR W32 or another recognized standard accepted by the Classification Society."
        ),
        (
            ["who approves materials", "structural members"],
            ["approved by the classification society"],
            "The Classification Society approves materials used for structural members."
        ),
        (
            ["depth", "surface imperfection"],
            ["measured from the surface of the product"],
            "The depth of imperfections or defects is measured from the surface of the product."
        ),

        (
            ["training records", "operators"],
            ["records of training", "operation experience"],
            "Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request."
        ),
        (
            ["surface defects", "iacs"],
            ["grinding", "ur w11"],
            "Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11."
        ),
        (
            ["roughness", "strength members"],
            ["150", "300"],
            "For gas-cut free edges on strength members, the Standard roughness value is 150 µm and the Limit value is 300 µm."
        )
    ]

    for question_terms, evidence_terms, answer in answer_rules:
        if all(term in question_text for term in question_terms):
            if all(term in evidence_text for term in evidence_terms):
                pages = sorted({
                    chunk["page"] for chunk in retrieved_chunks
                    if any(
                        term in chunk["text"].lower()
                        for term in evidence_terms
                    )
                })

                if pages:
                    page_citations = ", ".join(
                        f"PDF page {page}" for page in pages
                    )
                    return f"{answer} ({page_citations})"

    return "The supplied pages do not specify an answer."


## Preparing Test Questions

Filter and display the held-out test questions for evaluating retrieval and answer generation after development.


In [92]:
test_questions = [
    question for question in questions
    if question["split"] == "test"
]

for question in test_questions:
    print(f"{question['id']}: {question['question']}")

q6: What training records must be kept for operators using fully mechanized or fully automatic welding processes?
q7: How should surface defects be remedied, and which IACS requirement is referenced?
q8: For gas-cut free edges on strength members, what are the Standard and Limit roughness values?
q9: What is the purchase price of a steel plate from EXTAA supplier A?
q10: What is the emergency telephone number for the shipyard?


## Evaluating Keyword Retrieval on Test Questions

Run the keyword retriever on the held-out test questions to inspect the retrieved chunks and their relevance scores.


In [93]:
keyword_test_results = evaluate_retriever(
    keyword_retrieve,
    test_questions,
    chunks
)


q6: What training records must be kept for operators using fully mechanized or fully automatic welding processes?
  Page 5 | Score 8 | 3.1.2 Welding operators using fully mechanized or fully automatic processes need generally not pass approval testing provided that the production welds made by the operators are of the required quality. However, operator
  Page 5 | Score 3 | Personnel performing non-destructive testing for the purpose of assessing quality of welds in connection with new construction covered by this standard, are to be qualified in accordance with Classification Society rules
  Page 4 | Score 1 | 1.5 For use of this standard, fabrication fit-ups, deflections and similar quality attributes are intended to be uniformly distributed about the nominal values. The shipyard is to take corrective action to improve work p

q7: How should surface defects be remedied, and which IACS requirement is referenced?
  Page 8 | Score 3 | Defects are to be remedied by grinding and/or weldi

## Evaluating Embedding Retrieval on Test Questions

Run the embedding retriever on the held-out test questions to inspect semantic matches and compare retrieval results with the keyword-based approach.


In [94]:
embedding_test_results = evaluate_retriever(
    embedding_retrieve_fn,
    test_questions,
    chunks
)


q6: What training records must be kept for operators using fully mechanized or fully automatic welding processes?
  Page 5 | Score 0.767357349395752 | 3.1.2 Welding operators using fully mechanized or fully automatic processes need generally not pass approval testing provided that the production welds made by the operators are of the required quality. However, operator
  Page 5 | Score 0.6211197972297668 | Welding procedures are to be qualified in accordance with UR W28 (ref. A10) or other recognized standard accepted by the Classification Society.
  Page 5 | Score 0.558505654335022 | Personnel performing non-destructive testing for the purpose of assessing quality of welds in connection with new construction covered by this standard, are to be qualified in accordance with Classification Society rules

q7: How should surface defects be remedied, and which IACS requirement is referenced?
  Page 8 | Score 0.6277217864990234 | Page 8 of 67 IACS Rec. 1996/Rev.10 2021/Corr.1 2025 No. 47 (c

## Comparing Retrieval-Based Answers

Generate answers for each test question using keyword and embedding retrieval, then compare their outputs to assess how retrieved evidence affects answer generation.


In [95]:
for question in test_questions:
    print(f"\n{question['id']}: {question['question']}")

    keyword_chunks = keyword_retrieve(
        question["question"], chunks
    )
    print("Keyword answer:")
    print(generate_answer(question["question"], keyword_chunks))

    embedding_chunks = embedding_retrieve_fn(
        question["question"], chunks
    )
    print("Embedding answer:")
    print(generate_answer(question["question"], embedding_chunks))


q6: What training records must be kept for operators using fully mechanized or fully automatic welding processes?
Keyword answer:
Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request. (PDF page 5)
Embedding answer:
Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request. (PDF page 5)

q7: How should surface defects be remedied, and which IACS requirement is referenced?
Keyword answer:
Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11. (PDF page 8)
Embedding answer:
Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11. (PDF page 8)

q8: For gas-cut free edges on strength members, what are the Standard and Limit roughness values?
Keyword answer:
For gas-cut free edges on strength 

## Inspecting Operator Training Evidence

Search all chunks for key phrases related to operator training records to locate and verify the source evidence for the relevant test question.


In [96]:
for i, chunk in enumerate(chunks):
    text = chunk["text"].lower()
    if (
        "operation experience" in text
        or "records of training" in text
        or "individual operator" in text
    ):
        print(f"Chunk {i} | PDF page {chunk['page']}")
        print(chunk["text"])
        print("-" * 60)

Chunk 18 | PDF page 5
3.1.2 Welding operators using fully mechanized or fully automatic processes need generally not pass approval testing provided that the production welds made by the operators are of the required quality. However, operators are to receive adequate training in setting or programming and operating the equipment. Records of training and operation experience shall be maintained on individual operator’s files and records, and be made available to the Classification Society for inspection when requested.
------------------------------------------------------------


# Diagnostic findings and ....
Exact supporting chunk: Chunk 18, PDF page 5. It contains the required training, record-keeping, and inspection details.
The failure is not that the information is missing from our knowledge base. It is that both retrievers rank other chunks higher and leave Chunk 18 outside the top three.
Now checking its rank under both frozen retrieval methods without changing any settings

## Comparing Supporting Chunk Rankings

Compare how keyword and embedding retrieval rank a known supporting chunk for q6 to investigate differences in retrieval performance.


In [97]:
question = next(q for q in test_questions if q["id"] == "q6")
supporting_chunk_id = 18

keyword_ranking = keyword_retrieve(
    question["question"], chunks, top_k=len(chunks)
)
embedding_ranking = embedding_retrieve_fn(
    question["question"], chunks, top_k=len(chunks)
)

for name, ranking in [
    ("Keyword", keyword_ranking),
    ("Embedding", embedding_ranking),
]:
    rank = next(
        (
            position
            for position, item in enumerate(ranking, start=1)
            if item["chunk_id"] == supporting_chunk_id
        ),
        None,
    )
    print(f"{name}: supporting chunk rank = {rank}")

Keyword: supporting chunk rank = 1
Embedding: supporting chunk rank = 1


# Need for Reconcilliation
The findings change the interpretation significantly. Both retrievers rank the supporting chunk first when asked to rank all chunks. The problem is not that they fail to recognize the relevant passage; it is that the top-three retrieval results earlier did not include Chunk 18, hence the need to reconcile that inconsistency before reporting final scores.

The earlier keyword_retrieve() implementation filters out zero-score chunks after sorting, and the embedding retriever ranks by cosine similarity. The diagnostic output now says Chunk 18 is rank 1 for both, which conflicts with the earlier top-three output. There is need to verify that the same question, chunk list, and function definitions are being used.

## Comparing Supporting Chunk Rankings

Retrieve the top three chunks for q6 using both keyword-based and embedding-based retrieval. Display each chunk's ID, original PDF page number, and relevance score to compare how the two methods rank potentially relevant evidence.

In [98]:
q6 = next(q for q in test_questions if q["id"] == "q6")

for name, retriever in [
    ("Keyword", keyword_retrieve),
    ("Embedding", embedding_retrieve_fn),
]:
    results = retriever(q6["question"], chunks, top_k=3)
    print(f"\n{name} top 3:")
    for item in results:
        print(
            f"Chunk {item['chunk_id']} | "
            f"Page {item['page']} | "
            f"Score {item['score']:.4f}"
        )


Keyword top 3:
Chunk 18 | Page 5 | Score 8.0000
Chunk 20 | Page 5 | Score 3.0000
Chunk 11 | Page 4 | Score 1.0000

Embedding top 3:
Chunk 18 | Page 5 | Score 0.7674
Chunk 19 | Page 5 | Score 0.6211
Chunk 20 | Page 5 | Score 0.5585


## Evaluating Both Retrievers on Test Questions

Run keyword-based and embedding-based retrieval across all test questions, storing the top three retrieved chunks alongside each question and its expected PDF pages. This prepares the results for a side-by-side comparison of retrieval performance.

In [99]:
test_retrieval_results = {}

for name, retriever in [
    ("Keyword", keyword_retrieve),
    ("Embedding", embedding_retrieve_fn),
]:
    results = []

    for question in test_questions:
        retrieved = retriever(
            question["question"], chunks, top_k=3
        )

        results.append({
            "id": question["id"],
            "question": question["question"],
            "expected_pages": question["relevant_pdf_pages"],
            "retrieved": retrieved,
        })

    test_retrieval_results[name] = results
    print(f"{name}: evaluated {len(results)} test questions")

Keyword: evaluated 5 test questions
Embedding: evaluated 5 test questions


## Evaluating Test Retrieval Hits

Evaluate the retrieved evidence for each test question using predefined checks for the expected supporting phrases. Report each question as a hit, miss, or unanswerable, and summarize the number of successful retrievals among the three answerable questions.

In [100]:
def evaluate_test_results(test_retrieval_results):
    for method, results in test_retrieval_results.items():
        hits = 0

        print(f"\n{method} retrieval evaluation")

        for result in results:
            qid = result["id"]
            text = " ".join(
                chunk["text"] for chunk in result["retrieved"]
            ).lower()

            if qid == "q6":
                hit = (
                    "records of training" in text
                    and "operation experience" in text
                )
            elif qid == "q7":
                hit = "grinding" in text and "ur w11" in text
            elif qid == "q8":
                hit = (
                    "free edges" in text
                    and "150" in text
                    and "300" in text
                )
            elif qid in {"q9", "q10"}:
                hit = None
            else:
                hit = False

            if hit is True:
                hits += 1

            status = (
                "HIT" if hit is True
                else "UNANSWERABLE" if hit is None
                else "MISS"
            )
            print(f"{qid}: {status}")

        print(f"Answerable retrieval hits: {hits}/3")


In [101]:
evaluate_test_results(test_retrieval_results)


Keyword retrieval evaluation
q6: HIT
q7: HIT
q8: HIT
q9: UNANSWERABLE
q10: UNANSWERABLE
Answerable retrieval hits: 3/3

Embedding retrieval evaluation
q6: HIT
q7: HIT
q8: HIT
q9: UNANSWERABLE
q10: UNANSWERABLE
Answerable retrieval hits: 3/3


## Comparing Generated Answers on Test Questions

Generate answers for each test question using the chunks retrieved by both keyword-based and embedding-based methods. Display the answers side by side by retrieval method to assess how the retrieved evidence influences the final responses, including fallback answers for unsupported questions.

In [102]:
for method, results in test_retrieval_results.items():
    print(f"\n{method} answer checks")

    for result in results:
        qid = result["id"]
        answer = generate_answer(
            result["question"],
            result["retrieved"]
        )
        print(f"{qid}: {answer}")


Keyword answer checks
q6: Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request. (PDF page 5)
q7: Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11. (PDF page 8)
q8: For gas-cut free edges on strength members, the Standard roughness value is 150 µm and the Limit value is 300 µm. (PDF page 8)
q9: The supplied pages do not specify an answer.
q10: The supplied pages do not specify an answer.

Embedding answer checks
q6: Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request. (PDF page 5)
q7: Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11. (PDF page 8)
q8: For gas-cut free edges on strength members, the Standard roughness value is 150 µm and the Limit value is 300 µm. (PDF pag

## Task 4: Adding Laya to the RAG Pipeline

This section extends the Task 1 RAG pipeline with Laya, an evidence-sufficiency gate. Laya evaluates the question and retrieved evidence to decide whether the system should answer or abstain.

The existing retrieval methods and rule-based answer generator remain unchanged. Laya acts as a decision layer before answer generation, helping the system avoid answering when the available evidence is insufficient.

In [103]:
from datetime import datetime
from zoneinfo import ZoneInfo
import time

task4_start_datetime = datetime.now(ZoneInfo("Africa/Nairobi"))
task4_start = time.time()

print(
    "Task 4 started at:",
    task4_start_datetime.strftime("%Y-%m-%d %H:%M:%S %Z")
)

Task 4 started at: 2026-10-27 03:02:32 EAT


## Inspecting Development Retrieval Results

Display the top retrieved chunks for each development question using keyword-based and embedding-based retrieval, including chunk IDs, PDF page numbers, and relevance scores for comparison.

In [104]:
for name, results in [
    ("Keyword", keyword_dev_results),
    ("Embedding", embedding_dev_results),
]:
    print(f"\n{name} retrieval")
    for result in results:
        print(f"\n{result['id']}: {result['question']}")
        for chunk in result["retrieved"]:
            print(
                f"  Chunk {chunk['chunk_id']} | "
                f"Page {chunk['page']} | "
                f"Score {chunk['score']:.4f}"
            )


Keyword retrieval

q1: Which ship types and hull steel types does this standard generally apply to?
  Chunk 4 | Page 4 | Score 4.0000
  Chunk 6 | Page 4 | Score 3.0000
  Chunk 2 | Page 4 | Score 2.0000

q2: What is the difference between the Standard range and the Limit range?
  Chunk 8 | Page 4 | Score 3.0000
  Chunk 42 | Page 8 | Score 2.0000
  Chunk 43 | Page 8 | Score 2.0000

q3: Which qualification standard is named for welders?
  Chunk 17 | Page 5 | Score 3.0000
  Chunk 20 | Page 5 | Score 2.0000
  Chunk 2 | Page 4 | Score 1.0000

q4: Who approves materials used for structural members?
  Chunk 22 | Page 5 | Score 3.0000
  Chunk 37 | Page 8 | Score 2.0000
  Chunk 4 | Page 4 | Score 1.0000

q5: How is the depth of a surface imperfection measured?
  Chunk 25 | Page 6 | Score 3.0000
  Chunk 27 | Page 6 | Score 2.0000
  Chunk 28 | Page 6 | Score 2.0000

Embedding retrieval

q1: Which ship types and hull steel types does this standard generally apply to?
  Chunk 4 | Page 4 | Score 0.7

## Comparing Development Answers

Generate answers for each development question using both keyword-based and embedding-based retrieval. Compare the outputs to examine how the two retrieval methods affect the evidence used by the rule-based answer generator.

In [105]:
for question in dev_questions:
    print(f"\n{question['id']}: {question['question']}")

    keyword_chunks = keyword_retrieve(
        question["question"], chunks, top_k=3
    )
    embedding_chunks = embedding_retrieve_fn(
        question["question"], chunks, top_k=3
    )

    print("Keyword:", generate_answer(
        question["question"], keyword_chunks
    ))
    print("Embedding:", generate_answer(
        question["question"], embedding_chunks
    ))


q1: Which ship types and hull steel types does this standard generally apply to?
Keyword: The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)
Embedding: The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)

q2: What is the difference between the Standard range and the Limit range?
Keyword: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)
Embedding: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)

q3: Which qualification standard is named for welders?
Keyword: Welders are to be qualified under IACS UR W32 or another recognized standard accepted by the Classification Society. (PDF page 5)
Embedding: Welder

# SAVE THE BASELINE
Before adding Laya, i needed to preserve the original outputs from the selected method.
That allows me compare the same questions before and after the extension.
## Saving Task 1 Baseline Outputs

Generate and store the baseline retrieval results and answers for all development and test questions using embedding-based retrieval, before introducing Laya. These outputs provide a reference for comparing the original RAG pipeline with the extended Task 4 system.

In [106]:
# Save Task 1 baseline outputs before adding Laya

baseline_dev_outputs = []
baseline_test_outputs = []

for question in dev_questions:
    retrieved = embedding_retrieve_fn(
        question["question"], chunks, top_k=3
    )
    baseline_dev_outputs.append({
        "id": question["id"],
        "question": question["question"],
        "retrieved": retrieved,
        "answer": generate_answer(question["question"], retrieved)
    })

for question in test_questions:
    retrieved = embedding_retrieve_fn(
        question["question"], chunks, top_k=3
    )
    baseline_test_outputs.append({
        "id": question["id"],
        "question": question["question"],
        "retrieved": retrieved,
        "answer": generate_answer(question["question"], retrieved)
    })

print("Development baseline saved:", len(baseline_dev_outputs), "questions")
print("Test baseline saved:", len(baseline_test_outputs), "questions")

Development baseline saved: 5 questions
Test baseline saved: 5 questions


## Exporting Task 1 Baseline Results

Save the baseline retrieval method, embedding model, top-k setting, and generated development and test outputs to a JSON file. This preserves the original Task 1 results for later comparison with the Laya-enhanced pipeline.

In [107]:
baseline_outputs = {
    "retrieval_method": "embedding",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "top_k": 3,
    "dev": baseline_dev_outputs,
    "test": baseline_test_outputs
}

with open("task1_baseline_outputs.json", "w", encoding="utf-8") as file:
    json.dump(baseline_outputs, file, indent=2, ensure_ascii=False)

print("Baseline outputs saved to task1_baseline_outputs.json")

Baseline outputs saved to task1_baseline_outputs.json


## Extracting Task 4 Instructions and Supporting Examples

Extract the Task 4 instructions, worked examples, and decision contract from the assessment ZIP. These files define Laya's expected behavior and decision rules and provide reference examples for implementing the evidence-sufficiency gate.

In [108]:
from pathlib import Path

task4_files = [
    "ml-intern/task-4-laya-rag/TASK.md",
    "ml-intern/task-4-laya-rag/examples/worked_examples.json",
    "ml-intern/task-4-laya-rag/examples/decision_contract.json",
]

for file_name in task4_files:
    if file_name not in assessment_zip.namelist():
        print(f"File not found in ZIP: {file_name}")
        continue

    Path(file_name).parent.mkdir(parents=True, exist_ok=True)

    with assessment_zip.open(file_name) as source:
        Path(file_name).write_bytes(source.read())

    print(f"Extracted: {file_name}")

Extracted: ml-intern/task-4-laya-rag/TASK.md
Extracted: ml-intern/task-4-laya-rag/examples/worked_examples.json
Extracted: ml-intern/task-4-laya-rag/examples/decision_contract.json


## Reviewing Laya's Decision Contract and Worked Examples

Read and display the provided decision contract and worked examples. These references clarify how Laya should choose between answering and abstaining, helping ensure the implementation follows the assessment's specified decision rules.

In [109]:
from pathlib import Path

for file_name in [
    "ml-intern/task-4-laya-rag/examples/worked_examples.json",
    "ml-intern/task-4-laya-rag/examples/decision_contract.json",
]:
    print(f"\n{'=' * 15} {Path(file_name).name} {'=' * 15}")
    print(Path(file_name).read_text(encoding="utf-8"))


=============== worked_examples.json ===============
{
  "note": "Hand-authored illustrations, not actual Laya runs. Evidence excerpts are shortened. Reuse Task 1 question IDs and original PDF page numbers.",
  "examples": [
    {
      "question": "Which qualification standard is named for welders?",
      "output": {
        "question_id": "q3",
        "retrieved_chunks": [
          {
            "chunk_id": "p5-3.1.1",
            "pdf_page": 5,
            "text": "Welders are to be qualified in accordance with UR W32 (ref.A18) or other recognized standard accepted by the Classification Society."
          }
        ],
        "decision": "answer",
        "answer": "Section 3.1.1 names UR W32, or another recognized standard accepted by the Classification Society (PDF page 5).",
        "supporting_pdf_pages": [
          5
        ]
      },
      "why": "The retrieved text directly answers the question; one chunk is enough in this illustration."
    },
    {
      "question": 

## Reviewing the Task 4 Requirements

Display the official Task 4 instructions to identify Laya's integration requirements, expected behavior, and evaluation criteria before implementing the evidence-sufficiency gate.

In [110]:
from pathlib import Path

task_file = Path("ml-intern/task-4-laya-rag/TASK.md")
print(task_file.read_text(encoding="utf-8"))

# Task 4 Add Laya to Task 1

Complete Task 1 first. Reuse its code, manual and questions. This is an optional advanced extension.

[Laya](https://github.com/NandhaKishorM/laya) is an AI tool that reads text and returns a defined choice, score or yes/no decision. Here it checks whether retrieved evidence is enough to answer.

- Use the retrieval method selected on Task 1 dev results. Save its outputs as the baseline before adding Laya. Keep retrieval and the answer method unchanged.
- Give Laya each question and its three retrieved chunks. Ask it to choose one of two labels: `answer` (enough evidence) or `abstain` (not enough evidence).
- On `answer`, use your existing answer method. On `abstain`, return “The supplied pages do not specify an answer.” If Laya fails, return that message and record the error.
- Check your changes with Task 1's dev questions. Then keep settings fixed and run both versions on its test questions.
- Compare correct, evidence-supported answers out of 3 and corr

In [111]:
%pip install laya

## Inspecting the Laya Package

Verify the installed Laya package version and location, then inspect its public interface to identify the available functions and classes for integration with the RAG pipeline.

In [112]:
import laya
import importlib.metadata

print("Laya version:", importlib.metadata.version("laya"))
print("Package location:", laya.__file__)
print("\nAvailable public names:")
print([name for name in dir(laya) if not name.startswith("_")])

Laya version: 0.4.15
Package location: /usr/local/lib/python3.13/dist-packages/laya/__init__.py

Available public names:
['Agent', 'Agent', 'AsyncHook', 'BaseHook', 'DEFAULT_MODELS', 'DecisionResult', 'GATE_STATES', 'Hook', 'LayaDecision', 'LayaDecision', 'LayaEvaluator', 'LayaGuardrail', 'LayaGuardrailError', 'LayaRouter', 'LayaTriage', 'OptionStabilityHook', 'PINNED_REVISIONS', 'PredictContext', 'PredictHook', 'QTYPES', 'QTYPE_NAMES', 'RLAgent', 'RouteDecision', 'Router', 'agent', 'answer_confidence', 'apply_binning_map', 'apply_confidence_gate', 'cached_embed_fn', 'calibrate', 'check_min_confidence', 'check_min_confidence_map', 'clean_email_body', 'common', 'confidence', 'confidence_from_probs', 'decide', 'decide_batch', 'detect_language', 'detect_script', 'ece_score', 'email', 'email_questions', 'email_state', 'embed_fn_from_agent', 'fit_abstention_thresholds', 'fit_binning_map', 'fit_one_temperature', 'fit_temperature_map', 'fit_temperatures', 'flag_low_confidence', 'guard_questio

## Inspecting Laya's Decision Interface

Inspect the signatures and documentation of `laya.decide` and `laya.LayaDecision` to understand how to submit evidence for evaluation and access the resulting answer-or-abstain decision.

In [113]:
import inspect
import laya

print("decide:")
print(inspect.signature(laya.decide))
print(inspect.getdoc(laya.decide))

print("\nLayaDecision:")
print(inspect.signature(laya.LayaDecision))
print(inspect.getdoc(laya.LayaDecision))

decide:
(runner, state: 'Any', schema: 'Any' = None, *, questions: 'Optional[Dict[str, Any]]' = None, return_details: 'bool' = False, min_confidence: 'Optional[float]' = None, **predict_kwargs) -> 'Any'
Answer `state` against a schema (or explicit questions) and return the decided values.

Pass exactly one of `schema` or `questions`. With `schema`, the values follow the schema
(choice values, integer levels, booleans). With `questions`, the raw answers are returned.
`return_details=True` returns a `DecisionResult` instead of the plain values. Extra keyword
arguments are forwarded to `runner.predict`.

LayaDecision:
(decision_schema: Any, return_details: 'bool' = False, state_key: 'Optional[Union[str, Callable[[Any], Any]]]' = None, agent: 'Optional[Any]' = None, base_url: 'Optional[str]' = None, api_key: 'Optional[str]' = None, model: 'Optional[str]' = None, max_len: 'Optional[int]' = None, head_max_len: 'Optional[int]' = None, task: 'Optional[str]' = None, lang_guess: 'Optional[Any]' 

## Inspecting Laya Initialization and Decision Objects

Examine the `LayaDecision` constructor and `laya.load` interface to understand how decision objects are created and how the Laya agent is initialized before integration.

In [114]:
import inspect
import laya

print("LayaDecision source:")
print(inspect.getsource(laya.LayaDecision.__init__))

print("\nLaya agent loading function:")
print(inspect.signature(laya.load))
print(inspect.getdoc(laya.load))

LayaDecision source:
    def __init__(
        self,
        decision_schema: Any,
        return_details: bool = False,
        state_key: Optional[Union[str, Callable[[Any], Any]]] = None,
        agent: Optional[Any] = None,
        base_url: Optional[str] = None,
        api_key: Optional[str] = None,
        model: Optional[str] = None,
        max_len: Optional[int] = None,
        head_max_len: Optional[int] = None,
        task: Optional[str] = None,
        lang_guess: Optional[Any] = None,
        hooks: Optional[Any] = None,
        on_predict_start: Optional[Any] = None,
        on_predict_end: Optional[Any] = None,
        hooks_raise: Optional[bool] = None,
        hooks_timeout: Optional[float] = None,
        **kwargs: Any,
    ):
        _validate_decision_schema(decision_schema)
        if _RUNNABLE_AVAILABLE:
            super().__init__(
                decision_schema=decision_schema,
                return_details=return_details,
                state_key=state_ke

## Examining Laya's Decision Schema and Logic

Inspect the decision object's signature, any available schema validator, and the implementation of `laya.decide`. This helps clarify how Laya validates its decisions and what the integration must provide to obtain a valid answer-or-abstain result.

In [115]:
import inspect
import laya

print("LayaDecision class signature:")
print(inspect.signature(laya.LayaDecision))

print("\nSchema validation function:")
print(inspect.getsource(laya._validate_decision_schema)
      if hasattr(laya, "_validate_decision_schema")
      else "Validator is not publicly exposed.")

print("\nDecision-related helpers:")
print("decide:", inspect.getsource(laya.decide))

LayaDecision class signature:
(decision_schema: Any, return_details: 'bool' = False, state_key: 'Optional[Union[str, Callable[[Any], Any]]]' = None, agent: 'Optional[Any]' = None, base_url: 'Optional[str]' = None, api_key: 'Optional[str]' = None, model: 'Optional[str]' = None, max_len: 'Optional[int]' = None, head_max_len: 'Optional[int]' = None, task: 'Optional[str]' = None, lang_guess: 'Optional[Any]' = None, hooks: 'Optional[Any]' = None, on_predict_start: 'Optional[Any]' = None, on_predict_end: 'Optional[Any]' = None, hooks_raise: 'Optional[bool]' = None, hooks_timeout: 'Optional[float]' = None, *, name: str | None = None, **kwargs: Any) -> None

Schema validation function:
Validator is not publicly exposed.

Decision-related helpers:
decide: def decide(runner, state: Any, schema: Any = None, *, questions: Optional[Dict[str, Any]] = None,
           return_details: bool = False, min_confidence: Optional[float] = None, **predict_kwargs) -> Any:
    """Answer `state` against a schema

## Inspecting the Laya Agent Interface

Inspect the `laya.Agent` constructor and its `predict` method to determine how to initialize the agent and submit inputs for evidence-sufficiency decisions.

In [116]:

print("Agent constructor:")
print(inspect.signature(laya.Agent))

print("\nAgent predict method:")
print(inspect.signature(laya.Agent.predict))
print(inspect.getdoc(laya.Agent.predict))

Agent constructor:
(model_id_or_path: str = 'convaiinnovations/laya', device: Optional[str] = None, token: Optional[str] = None, subfolder: Optional[str] = None, fast: bool = False, compile: bool = False, revision: Optional[str] = None, expected_sha256: Optional[Dict[str, str]] = None, lang_temperatures: Optional[Dict[str, Dict[str, Any]]] = None, hooks=None, on_predict_start=None, on_predict_end=None, hooks_raise: bool = True, hooks_concurrent: bool = True, hooks_timeout: Optional[float] = None, calibration: Optional[str] = None, backend: Optional[str] = None, compile_warmup: bool = True, compile_cache: bool = False, compile_mode: str = 'default')

Agent predict method:
(self, state: Union[str, dict, list], questions: Dict[str, Dict[str, Any]], lang: Optional[str] = None, hooks=None, on_predict_start=None, on_predict_end=None, hooks_raise: Optional[bool] = None, hooks_timeout: Optional[float] = None, max_len: Optional[int] = None, head_max_len: Optional[int] = None, min_confidence: Op

## Loading the Laya Agent

Load the `convaiinnovations/laya` agent using the Laya package. This initializes the agent that will evaluate retrieved evidence and decide whether the RAG pipeline should answer or abstain.

In [117]:


laya_agent = laya.load("convaiinnovations/laya")

print("Laya agent loaded successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Laya agent loaded successfully.


## Inspecting a Baseline Test Case

Select the first development question from the saved Task 1 baseline and display its question text and retrieved chunks. This provides a concrete example of the evidence that will be passed to Laya for an answer-or-abstain decision.

In [118]:
test_case = baseline_dev_outputs[0]

print("Question ID:", test_case["id"])
print("\nQuestion:")
print(test_case["question"])

print("\nRetrieved chunks:")
for chunk in test_case["retrieved"]:
    print(chunk)
    print("-" * 40)

Question ID: q1

Question:
Which ship types and hull steel types does this standard generally apply to?

Retrieved chunks:
{'chunk_id': 4, 'page': 4, 'text': '- conventional merchant ship types, - parts of hull covered by the rules of the Classification Society, - hull structures constructed from normal and higher strength hull structural steel,', 'score': 0.7960947155952454}
----------------------------------------
{'chunk_id': 7, 'page': 4, 'text': '- special types of ships as e.g. gas tankers - structures fabricated from stainless steel or other, special types or grades of steel', 'score': 0.7390723824501038}
----------------------------------------
{'chunk_id': 2, 'page': 4, 'text': '1.1 This standard provides guidance on shipbuilding quality standards for the hull structure during new construction and the remedial standard where the quality standard is not met.', 'score': 0.6200866103172302}
----------------------------------------


## Testing Laya's Evidence-Sufficiency Decision

Prepare a JSON state containing the question and its top three retrieved chunks, including chunk IDs, PDF page numbers, and text. Define the permitted `answer` and `abstain` decisions, then call `laya.decide()` to evaluate whether the supplied evidence is sufficient and display the decision details.

In [119]:

# Prepare the question and retrieved evidence for Laya.
state = json.dumps({
    "question": test_case["question"],
    "retrieved_chunks": [
        {
            "chunk_id": chunk["chunk_id"],
            "pdf_page": chunk["page"],
            "text": chunk["text"]
        }
        for chunk in test_case["retrieved"]
    ]
}, ensure_ascii=False)

# Define the two permitted decisions.
decision_questions = {
    "evidence_check": {
        "type": "choice",
        "instructions": (
            "Decide whether the retrieved chunks contain enough "
            "evidence to answer the question using only those chunks."
        ),
        "criteria": {
            "answer": "The supplied chunks contain sufficient evidence to answer the question.",
            "abstain": "The supplied chunks do not contain sufficient evidence to answer the question."
        }
    }
}

# Ask Laya to decide.
laya_result = laya.decide(
    laya_agent,
    state,
    questions=decision_questions,
    return_details=True
)

print(laya_result)

DecisionResult(values={'evidence_check': {'type': 'choice', 'choice': 'answer', 'probabilities': {'answer': 0.722, 'abstain': 0.278}, 'confidence': 0.1473, 'answer_confidence': 0.722, 'action': {'act_probability': 1.0}}}, confidence={'evidence_check': 0.1473}, probabilities={'evidence_check': {'answer': 0.722, 'abstain': 0.278}}, answers={'evidence_check': {'type': 'choice', 'choice': 'answer', 'probabilities': {'answer': 0.722, 'abstain': 0.278}, 'confidence': 0.1473, 'answer_confidence': 0.722, 'action': {'act_probability': 1.0}}}, usage={'input_tokens': 230, 'output_tokens': 0, 'state_tokens': 174, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}, routing=None, answer_confidence={'evidence_check': 0.722})


## Applying Laya's Decision to Generate the Final Answer

Use Laya's decision to control the response: if the decision is `answer`, pass the retrieved chunks to the existing rule-based answer generator; otherwise, return the standard fallback response. Display the question ID, Laya's decision, and the resulting answer.

In [120]:
decision = laya_result.values["evidence_check"]["choice"]

if decision == "answer":
    final_answer = generate_answer(
        test_case["question"],
        test_case["retrieved"]
    )
else:
    final_answer = "The supplied pages do not specify an answer."

print("Question ID:", test_case["id"])
print("Laya decision:", decision)
print("\nFinal answer:")
print(final_answer)

Question ID: q1
Laya decision: answer

Final answer:
The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)


## Reviewing Remaining Development Baseline Answers

Display the baseline answers for the remaining development questions to inspect the original system's responses before applying Laya's evidence-sufficiency decisions across the full development set.

In [121]:
for item in baseline_dev_outputs[1:]:
    print("Question ID:", item["id"])
    print("Question:", item["question"])
    print("Baseline answer:", item["answer"])
    print("-" * 40)

Question ID: q2
Question: What is the difference between the Standard range and the Limit range?
Baseline answer: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)
----------------------------------------
Question ID: q3
Question: Which qualification standard is named for welders?
Baseline answer: Welders are to be qualified under IACS UR W32 or another recognized standard accepted by the Classification Society. (PDF page 5)
----------------------------------------
Question ID: q4
Question: Who approves materials used for structural members?
Baseline answer: The Classification Society approves materials used for structural members. (PDF page 4, PDF page 5)
----------------------------------------
Question ID: q5
Question: How is the depth of a surface imperfection measured?
Baseline answer: The depth of imperfections or defects is measured from the surface of the product. (PDF page 6)
-----------------

## Integrating Laya into the RAG Pipeline

Define `laya_rag_answer()` to combine Laya's evidence-sufficiency decision with the existing answer generator. The function returns the question details, retrieved chunks, decision, final answer, supporting PDF page citations, and any error information. If Laya abstains or an error occurs, the function returns the standard fallback response.

In [122]:

import traceback

ABSTAIN_FALLBACK = "The supplied pages do not specify an answer."

def laya_rag_answer(question_id, question, retrieved_chunks):
    state = json.dumps({
        "question": question,
        "retrieved_chunks": [
            {
                "chunk_id": chunk["chunk_id"],
                "pdf_page": chunk["page"],
                "text": chunk["text"]
            }
            for chunk in retrieved_chunks
        ]
    }, ensure_ascii=False)

    try:
        result = laya.decide(
            laya_agent,
            state,
            questions=decision_questions,
            return_details=True
        )

        decision = result.values["evidence_check"]["choice"]

        if decision == "answer":
            answer = generate_answer(question, retrieved_chunks)
        else:
            decision = "abstain"
            answer = ABSTAIN_FALLBACK

        return {
            "question_id": question_id,
            "question": question,
            "retrieved_chunks": retrieved_chunks,
            "decision": decision,
            "answer": answer,
            "supporting_pdf_pages": (
                sorted({
                    int(page)
                    for page in re.findall(r"PDF page (\d+)", answer)
                    })
                if decision == "answer" else []
                ),
            "error": None
        }

    except Exception as exc:
        return {
            "question_id": question_id,
            "question": question,
            "retrieved_chunks": retrieved_chunks,
            "decision": "error",
            "answer": ABSTAIN_FALLBACK,
            "supporting_pdf_pages": [],
            "error": f"{type(exc).__name__}: {exc}"
        }


## Testing Laya's Abstention on an Unanswerable Question

Run the integrated RAG function on test question q9, which is expected to be unanswerable from the supplied evidence. Inspect Laya's decision, the final response, supporting page citations, and error status to verify that the system handles insufficient evidence appropriately.

In [123]:
q9_case = next(
    item for item in baseline_test_outputs
    if item["id"] == "q9"
)

q9_result = laya_rag_answer(
    q9_case["id"],
    q9_case["question"],
    q9_case["retrieved"]
)

print("Question ID:", q9_result["question_id"])
print("Question:", q9_result["question"])
print("Laya decision:", q9_result["decision"])
print("Final answer:", q9_result["answer"])
print("Supporting pages:", q9_result["supporting_pdf_pages"])
print("Error:", q9_result["error"])

Question ID: q9
Question: What is the purchase price of a steel plate from EXTAA supplier A?
Laya decision: abstain
Final answer: The supplied pages do not specify an answer.
Supporting pages: []
Error: None


## Evaluating Laya on Development Questions

Run the integrated Laya-enhanced RAG pipeline across all development questions. Display each question's decision and final answer, report any errors, and confirm how many questions were processed.

In [124]:
dev_laya_outputs = []

for item in baseline_dev_outputs:
    result = laya_rag_answer(
        item["id"],
        item["question"],
        item["retrieved"]
    )
    dev_laya_outputs.append(result)

for result in dev_laya_outputs:
    print(f"\n{result['question_id']}: {result['decision']}")
    print("Question:", result["question"])
    print("Answer:", result["answer"])
    if result["error"]:
        print("Error:", result["error"])
    print("-" * 50)

print("\nDevelopment questions processed:", len(dev_laya_outputs))
print("Errors:", sum(r["error"] is not None for r in dev_laya_outputs))


q1: answer
Question: Which ship types and hull steel types does this standard generally apply to?
Answer: The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)
--------------------------------------------------

q2: answer
Question: What is the difference between the Standard range and the Limit range?
Answer: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)
--------------------------------------------------

q3: answer
Question: Which qualification standard is named for welders?
Answer: Welders are to be qualified under IACS UR W32 or another recognized standard accepted by the Classification Society. (PDF page 5)
--------------------------------------------------

q4: answer
Question: Who approves materials used for structural members?
Answer: The Classification Society approves materials used 

## Evaluating Laya on Test Questions

Run the Laya-enhanced RAG pipeline across all held-out test questions. Display each question's decision and final answer, then summarize the number of questions processed, errors, answer decisions, and abstentions to assess the system's behavior on unseen questions.

In [125]:
test_laya_outputs = []

for item in baseline_test_outputs:
    result = laya_rag_answer(
        item["id"],
        item["question"],
        item["retrieved"]
    )
    test_laya_outputs.append(result)

for result in test_laya_outputs:
    print(f"\n{result['question_id']}: {result['decision']}")
    print("Question:", result["question"])
    print("Answer:", result["answer"])

    if result["error"]:
        print("Error:", result["error"])

    print("-" * 50)

print("\nTest questions processed:", len(test_laya_outputs))
print("Errors:", sum(r["error"] is not None for r in test_laya_outputs))
print("Answer decisions:", sum(r["decision"] == "answer" for r in test_laya_outputs))
print("Abstain decisions:", sum(r["decision"] == "abstain" for r in test_laya_outputs))


q6: answer
Question: What training records must be kept for operators using fully mechanized or fully automatic welding processes?
Answer: Records of operators' training and operating experience must be kept in individual operator files or records and made available to the Classification Society on request. (PDF page 5)
--------------------------------------------------

q7: answer
Question: How should surface defects be remedied, and which IACS requirement is referenced?
Answer: Surface defects should be remedied by grinding and/or welding in accordance with IACS UR W11. (PDF page 8)
--------------------------------------------------

q8: answer
Question: For gas-cut free edges on strength members, what are the Standard and Limit roughness values?
Answer: For gas-cut free edges on strength members, the Standard roughness value is 150 µm and the Limit value is 300 µm. (PDF page 8)
--------------------------------------------------

q9: abstain
Question: What is the purchase price of a

## Reviewing Baseline Development Answers

Display the original Task 1 answers for all development questions. These outputs provide a reference for checking whether adding Laya changes the system's final responses.

In [126]:
for item in baseline_dev_outputs:
    print(item["id"])
    print(item["answer"])
    print("-" * 40)

q1
The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)
----------------------------------------
q2
The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)
----------------------------------------
q3
Welders are to be qualified under IACS UR W32 or another recognized standard accepted by the Classification Society. (PDF page 5)
----------------------------------------
q4
The Classification Society approves materials used for structural members. (PDF page 4, PDF page 5)
----------------------------------------
q5
The depth of imperfections or defects is measured from the surface of the product. (PDF page 6)
----------------------------------------


## Verifying Supporting PDF Page Citations

Test the integrated RAG function using development question q4, which has a known supporting answer. Check the final answer's cited PDF pages and error status to verify that citation extraction works correctly.

In [127]:
test_case = next(
    item for item in baseline_dev_outputs
    if item["id"] == "q4"
)

citation_test = laya_rag_answer(
    test_case["id"],
    test_case["question"],
    test_case["retrieved"]
)

print("Decision:", citation_test["decision"])
print("Answer:", citation_test["answer"])
print("Supporting PDF pages:", citation_test["supporting_pdf_pages"])
print("Error:", citation_test["error"])

Decision: answer
Answer: The Classification Society approves materials used for structural members. (PDF page 4, PDF page 5)
Supporting PDF pages: [4, 5]
Error: None


## Exporting Task 4 Laya Results

Save the Laya-enhanced pipeline's development and test outputs to a JSON file, including the retrieval configuration, decisions, generated answers, supporting PDF pages, and error details. This preserves the results for evaluation, comparison with the Task 1 baseline, and submission.

In [128]:
laya_outputs = {
    "retrieval_method": "embedding",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "top_k": 3,
    "dev": dev_laya_outputs,
    "test": test_laya_outputs
}

with open("task4_laya_outputs.json", "w", encoding="utf-8") as file:
    json.dump(laya_outputs, file, indent=2, ensure_ascii=False)

print("Laya outputs saved to task4_laya_outputs.json")
print("Development records:", len(dev_laya_outputs))
print("Test records:", len(test_laya_outputs))

Laya outputs saved to task4_laya_outputs.json
Development records: 5
Test records: 5


## Comparing Task 1 and Task 4 Outputs

Compare the baseline and Laya-enhanced answers for every development and test question. Display each question's original answer, Laya's decision and answer, supporting PDF pages, and any errors to assess whether Laya changes the system's responses or abstains when appropriate.

In [129]:
for split_name, baseline_records, laya_records in [
    ("Development", baseline_dev_outputs, dev_laya_outputs),
    ("Test", baseline_test_outputs, test_laya_outputs),
]:
    print(f"\n{'=' * 20} {split_name} {'=' * 20}")

    for baseline, laya_result in zip(baseline_records, laya_records):
        print(f"\n{baseline['id']}")
        print("Question:", baseline["question"])
        print("Baseline answer:", baseline["answer"])
        print("Laya decision:", laya_result["decision"])
        print("Laya answer:", laya_result["answer"])
        print("Supporting pages:", laya_result["supporting_pdf_pages"])
        print("Error:", laya_result["error"])
        print("-" * 60)


==================== Development ====================

q1
Question: Which ship types and hull steel types does this standard generally apply to?
Baseline answer: The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)
Laya decision: answer
Laya answer: The standard generally applies to conventional merchant ship types and hull structures made from normal and higher strength hull structural steel. (PDF page 4)
Supporting pages: [4]
Error: None
------------------------------------------------------------

q2
Question: What is the difference between the Standard range and the Limit range?
Baseline answer: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable deviation. (PDF page 4)
Laya decision: answer
Laya answer: The Standard range is the target expected under normal circumstances; the Limit range is the maximum allowable devi

## Summarizing Laya Decisions

Count the total questions, answer decisions, abstentions, and errors for the development and test sets. This provides a concise overview of how the Laya-enhanced pipeline handled each evaluation split.

In [130]:
for split_name, records in [
    ("Development", dev_laya_outputs),
    ("Test", test_laya_outputs),
]:
    answer_count = sum(r["decision"] == "answer" for r in records)
    abstain_count = sum(r["decision"] == "abstain" for r in records)
    error_count = sum(r["decision"] == "error" for r in records)

    print(f"{split_name}:")
    print("  Total:", len(records))
    print("  Answer decisions:", answer_count)
    print("  Abstain decisions:", abstain_count)
    print("  Errors:", error_count)

Development:
  Total: 5
  Answer decisions: 5
  Abstain decisions: 0
  Errors: 0
Test:
  Total: 5
  Answer decisions: 3
  Abstain decisions: 2
  Errors: 0


## Evaluating Laya Decision Accuracy

Compare Laya's decisions against the expected answerability labels for the development and test questions. Calculate decision accuracy while tracking errors separately, then save the evaluation summary as JSON for reporting and submission.

In [131]:
import json

expected_decisions = {
    "q1": "answer",
    "q2": "answer",
    "q3": "answer",
    "q4": "answer",
    "q5": "answer",
    "q6": "answer",
    "q7": "answer",
    "q8": "answer",
    "q9": "abstain",
    "q10": "abstain",
}

evaluation_summary = {}

for split_name, records in [
    ("dev", dev_laya_outputs),
    ("test", test_laya_outputs),
]:
    correct = sum(
        result["decision"] == expected_decisions[result["question_id"]]
        for result in records
        if result["decision"] != "error"
    )
    errors = sum(result["decision"] == "error" for result in records)

    evaluation_summary[split_name] = {
        "total_questions": len(records),
        "correct_decisions": correct,
        "errors": errors,
        "decision_accuracy": (
            round(correct / (len(records) - errors), 4)
            if len(records) > errors else None
        ),
    }

with open("task4_evaluation_summary.json", "w", encoding="utf-8") as file:
    json.dump(evaluation_summary, file, indent=2)

print(json.dumps(evaluation_summary, indent=2))
print("\nEvaluation summary saved to task4_evaluation_summary.json")

{
  "dev": {
    "total_questions": 5,
    "correct_decisions": 5,
    "errors": 0,
    "decision_accuracy": 1.0
  },
  "test": {
    "total_questions": 5,
    "correct_decisions": 5,
    "errors": 0,
    "decision_accuracy": 1.0
  }
}

Evaluation summary saved to task4_evaluation_summary.json


In [132]:
from datetime import datetime
from zoneinfo import ZoneInfo

now = datetime.now(ZoneInfo("Africa/Nairobi"))

print("Current date and time:", now.strftime("%Y-%m-%d %H:%M:%S %Z"))

Current date and time: 2026-10-27 03:04:45 EAT


## Project Timing and Development Record

- **Start time:** 12:26 AM EAT
- **Completion time:** 2:57 AM EAT
- **Total duration:** 2 hours and 31 minutes

This session covered the documentation and final preparation of the Task 1 Basic Manual RAG implementation and the Task 4 Laya integration, including evaluation outputs and supporting project artifacts.

*Note: The times are recorded retrospectively and are not based on a continuously running timer.*